# 동아대 QML 강의 — 준비 ① : 16×16 전처리 npy 재생성 (Colab 전용)

**목적**: 2026-10-01 커리큘럼 재설계로 실습 해상도가 8×8 → **16×16**으로 올라가면서 기존 전처리 자산이 무효가 됨. 원본 HAM10000 이미지가 필요한 단계라 **Kaggle 인증이 가능한 Colab에서만 실행 가능**하고, 이후 ②(QGAN 학습)·③(MediQ-GAN 사전학습)은 전부 로컬에서 돌 수 있음.

**이 노트북이 하는 일만**: kagglehub 다운로드 → 세그멘테이션 전처리 → `16×16` npy 2개 저장 → zip으로 내보내기. 학습은 하지 않음.

**결과물**: `step1_16px_assets.zip` (안에 `df_preprocessed.npy`, `nv_preprocessed.npy`, `manifest.json`)

**예상 소요시간**: 다운로드 약 6GB(실측 필요) + 전처리 460장 약 5~15분

**끝나고 할 일**: 내려받은 zip을 풀어 `day3/assets/`에 덮어쓰고 커밋·푸시 → 그 다음 ②③은 로컬에서 실행


## 1. 설치 · 설정

`scikit-fuzzy`는 2일차 세그멘테이션(K-means + Fuzzy C-means)에 필요함. cv2·sklearn은 Colab에 이미 있음.

In [ ]:
!pip install -q kagglehub scikit-fuzzy
print("설치 완료")

In [ ]:
import os, json, time, zipfile
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import skfuzzy as fuzz

# ── 설정 ─────────────────────────────────────────────────────────────
IMG_SIZE = 16     # 2026-10-01 확정 스펙(2·3일차 노트북과 반드시 동일해야 함)
SEG_SIZE = (64, 64)   # 세그멘테이션 작업 해상도(기존과 동일)
N_DF = None       # None = DF 전체(115장)
N_NV = 345        # NV는 DF의 3배만 사용(기존 자산과 동일 기준 — 비교 가능성 유지)
SAVE_DIR = "/content/step1_assets"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"IMG_SIZE={IMG_SIZE} / NV {N_NV}장 / 저장 위치 {SAVE_DIR}")

## 2. HAM10000 다운로드 (kagglehub)

인증 오류가 나면 아래 셀의 `kagglehub.login()` 주석을 풀고 토큰을 입력할 것(Kaggle → Settings → API → Create New Token).

⚠ 약 6GB가 **Colab 런타임 캐시**에만 저장됨(개인 PC·Drive로 가지 않음). 런타임을 초기화하면 다시 받아야 함.

In [ ]:
import kagglehub
# kagglehub.login()   # 인증 오류 시에만 주석 해제

_t0 = time.time()
DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print(f"다운로드 소요시간: {(time.time()-_t0)/60:.1f}분  (← 2일차 수업 시간 배분에 그대로 반영할 숫자)")
print("경로:", DATA_DIR)

metadata = pd.read_csv(f"{DATA_DIR}/HAM10000_metadata.csv")
counts = metadata["dx"].value_counts()
print(f"\n전체 {len(metadata)}장 | DF {counts.get('df',0)}장 / NV {counts.get('nv',0)}장")

## 3. 전처리 파이프라인 (2일차 학생용 노트북과 **동일 로직**)

여기 코드가 학생이 수업에서 돌릴 코드와 달라지면 강사 체크포인트와 학생 결과가 어긋나므로, `day3/data_pipeline.py`의 `segment_fcm_kmeans`·`resize_for_encoding`을 그대로 인라인함.

In [ ]:
def segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=None):
    """HAM10000 QGAN 논문 Section 3.1 — K-means 1차 클러스터링 후 Fuzzy C-means로 정제."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if resize_to is not None:
        gray = cv2.resize(gray, resize_to, interpolation=cv2.INTER_AREA)
    h, w = gray.shape
    pixels = gray.reshape(-1, 1).astype(np.float32)

    km = KMeans(n_clusters=n_clusters, n_init=4, random_state=0).fit(pixels)
    dist = np.abs(pixels - km.cluster_centers_.reshape(1, -1)) + 1e-6
    u_init = 1.0 / dist ** 2
    u_init = (u_init / u_init.sum(axis=1, keepdims=True)).T

    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        pixels.T, c=n_clusters, m=2.0, error=1e-4, maxiter=100, init=u_init
    )
    membership = np.argmax(u, axis=0)
    lesion_cluster = np.argmin(cntr.flatten())
    mask = (membership == lesion_cluster).reshape(h, w).astype(np.uint8) * 255
    return cv2.bitwise_and(gray, gray, mask=mask)


def resize_for_encoding(img_gray, size):
    return cv2.resize(img_gray, size, interpolation=cv2.INTER_AREA)


def find_image_path(image_id):
    for sub in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        p = f"{DATA_DIR}/{sub}/{image_id}.jpg"
        if os.path.exists(p):
            return p
    return None

print("전처리 함수 준비 완료")

### 3-1. 1장으로 먼저 확인 + 전체 소요시간 추정

In [ ]:
row = metadata[metadata["dx"] == "df"].iloc[0]
img_bgr = cv2.imread(find_image_path(row["image_id"]))

t0 = time.time()
seg = segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=SEG_SIZE)
per_img = time.time() - t0
small = resize_for_encoding(seg, (IMG_SIZE, IMG_SIZE))

n_total = (N_DF or int(metadata["dx"].value_counts().get("df", 0))) + N_NV
print(f"1장 소요 {per_img:.2f}초 → 전체 {n_total}장 예상 {per_img*n_total/60:.1f}분")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title("원본"); axes[0].axis("off")
axes[1].imshow(seg, cmap="gray"); axes[1].set_title(f"세그멘테이션 {SEG_SIZE[0]}x{SEG_SIZE[1]}"); axes[1].axis("off")
axes[2].imshow(small, cmap="gray"); axes[2].set_title(f"인코딩용 {IMG_SIZE}x{IMG_SIZE}"); axes[2].axis("off")
plt.tight_layout(); plt.show()
print("→ 세 번째 그림에 병변 형태가 남아 있어야 정상. 전부 검게 나오면 세그멘테이션 실패이므로 중단하고 확인할 것")

## 4. DF·NV 전체 전처리 → npy 저장

값 범위는 **0~1**로 저장함(척도 규약). 중간에 끊기면 그냥 이 셀만 다시 실행하면 됨.

In [ ]:
def build_dataset(target_class, img_size, max_n=None):
    df = metadata[metadata["dx"] == target_class]
    if max_n:
        df = df.head(max_n)
    imgs, skipped = [], 0
    t0 = time.time()
    for i, (_, r) in enumerate(df.iterrows(), 1):
        path = find_image_path(r["image_id"])
        if path is None:
            skipped += 1
            continue
        seg = segment_fcm_kmeans(cv2.imread(path), n_clusters=4, resize_to=SEG_SIZE)
        imgs.append(resize_for_encoding(seg, (img_size, img_size)).astype(np.float32) / 255.0)
        if i % 25 == 0:
            print(f"  {target_class} {i}/{len(df)}장 ({(time.time()-t0)/60:.1f}분 경과)", flush=True)
    print(f"{target_class}: {len(imgs)}장 완료 (누락 {skipped}장, {(time.time()-t0)/60:.1f}분)")
    return np.stack(imgs) if imgs else np.zeros((0, img_size, img_size), dtype=np.float32)


df_imgs = build_dataset("df", IMG_SIZE, N_DF)
nv_imgs = build_dataset("nv", IMG_SIZE, N_NV)

np.save(f"{SAVE_DIR}/df_preprocessed.npy", df_imgs)
np.save(f"{SAVE_DIR}/nv_preprocessed.npy", nv_imgs)
print(f"\n저장 완료: df {df_imgs.shape} / nv {nv_imgs.shape}")

## 5. 검증 — 이 셀이 통과해야 저장소에 커밋할 것

In [ ]:
ok = True
for name, arr in [("df", df_imgs), ("nv", nv_imgs)]:
    cond = (arr.ndim == 3 and arr.shape[1] == IMG_SIZE and arr.shape[2] == IMG_SIZE
            and arr.min() >= -1e-6 and arr.max() <= 1 + 1e-6 and len(arr) > 0)
    print(f"[{'OK ' if cond else 'FAIL'}] {name}: shape {arr.shape}, 범위 {arr.min():.3f}~{arr.max():.3f}, "
          f"평균 {arr.mean():.3f}")
    ok = ok and cond
print("\n전체 검증:", "통과" if ok else "실패 — 커밋하지 말고 위 단계 확인할 것")

# 클래스 평균 이미지 — 지표 ②(클래스평균 상관계수)의 기준이 되는 그림
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(df_imgs.mean(axis=0), cmap="gray"); axes[0].set_title("DF 평균"); axes[0].axis("off")
axes[1].imshow(nv_imgs.mean(axis=0), cmap="gray"); axes[1].set_title("NV 평균"); axes[1].axis("off")
axes[2].imshow(df_imgs[0], cmap="gray"); axes[2].set_title("DF 샘플"); axes[2].axis("off")
axes[3].imshow(nv_imgs[0], cmap="gray"); axes[3].set_title("NV 샘플"); axes[3].axis("off")
plt.tight_layout(); plt.show()
print("→ DF 평균과 NV 평균이 눈에 띄게 달라야 지표 ②가 두 클래스를 구분할 수 있음")

## 6. zip으로 내보내기

다운로드가 안 되면(브라우저 차단 등) 왼쪽 파일 탐색기에서 `/content/step1_16px_assets.zip`을 직접 내려받으면 됨.

In [ ]:
manifest = {
    "img_size": IMG_SIZE, "seg_size": list(SEG_SIZE),
    "df_count": int(len(df_imgs)), "nv_count": int(len(nv_imgs)),
    "scale": "0~1", "source": "kmader/skin-cancer-mnist-ham10000",
    "note": "2026-10-01 커리큘럼 재설계(16x16)용 재생성 자산 — 재생산 체인 ①",
}
with open(f"{SAVE_DIR}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

zip_path = "/content/step1_16px_assets.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in ["df_preprocessed.npy", "nv_preprocessed.npy", "manifest.json"]:
        z.write(f"{SAVE_DIR}/{fn}", arcname=fn)
print(f"{zip_path} 생성 ({os.path.getsize(zip_path)/1024:.0f} KB)")

from google.colab import files
files.download(zip_path)

## 7. 끝난 다음 (로컬에서)

1. zip을 풀어 `day3/assets/`의 `df_preprocessed.npy`·`nv_preprocessed.npy`를 **덮어쓰기** + `manifest.json` 추가 → 커밋·푸시
   - 기존 8×8 자산을 되돌려야 할 경우를 위해 git 이력에 남으므로 별도 백업은 불필요
2. ② QGAN 체크포인트 학습 (약 20분)

```
cd day3
C:/qmlvenv/Scripts/python.exe -X utf8 train_qgan_checkpoint.py --npy assets/df_preprocessed.npy --out_dir ./checkpoints --epochs 600
```

3. ③ MediQ-GAN 사전학습 (약 3시간, 밤샘 권장)

```
C:/qmlvenv/Scripts/python.exe -X utf8 train_mediqgan_checkpoint.py --npy assets/df_preprocessed.npy --out_dir ./checkpoints --epochs 30 --save_every 5
```

4. 생성된 `qgan_df_16px.pt`·`mediqgan_final.pt`·`mediqgan_e010.pt`(중간본)을 `day3/assets/`로 옮겨 커밋
